# Cloud Kitchen P&L — Data Analysis

**Author:** Shubhjeet Paul &nbsp;•&nbsp; **Submission:** Rebel Foods
This notebook complements `app.py` (the Streamlit dashboards). It walks through the analysis:
1. Load and inspect the data
2. Build derived metrics (GM %, CM, CM %, EBITDA %, Variance %)
3. Construct the buckets required by Dashboard 2
4. Reproduce both dashboard tables
5. Extra business insights

I created a seprate file `data_prep.py` for data preprocessing of MONTH_DT,GM %,,CM,CM %,EBITDA %,VARIANCE %,	REVENUE BUCKET,VARIANCE BUCKET.	


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from data_prep import load_and_prepare, REVENUE_BUCKETS

pd.set_option('display.float_format', '{:,.2f}'.format)
df = load_and_prepare('Kittchen_PNL_Data.xlsx')
df.shape

(2100, 26)

## 1. Data overview

Here, I create a overview of all the date range, stores, cities, zone which we have in our dataset.

Also added first 5 rows to understand what data we have using `df.head` function.

* Used `strftime` to format the datetime to display the dates.
* Used `nunique` function to find the number of unique stores from the dataset.
* Used `unique` function to get the names of the city.

In [3]:
print('Date range :', df['MONTH_DT'].min().strftime('%b %Y'), '→', df['MONTH_DT'].max().strftime('%b %Y'))
print('Stores     :', df['STORE'].nunique())
print('Cities     :', df['CITY'].nunique(), '—', sorted(df['CITY'].unique()))
print('Zones      :', sorted(df['ZONE MAPPING'].unique()))
df.head()

Date range : Oct 2023 → Mar 2024
Stores     : 344
Cities     : 5 — ['Ahmedabad', 'Bangalore', 'Hyderabad', 'Mumbai', 'Pune']
Zones      : ['East', 'North', 'South', 'West']


,MONTH,CITY,STORE,STATUS,ZONE MAPPING,ORDER COUNT,CART SALES,DISCOUNT,NET REVENUE,IDEAL FOOD COST,...,EBITDA COHORT,MONTH_DT,GM %,CM,CM %,EBITDA %,VARIANCE %,REVENUE BUCKET,VARIANCE BUCKET,MONTH LABEL
0,Oct-2023,Bangalore,Acevedo Inc,Inactive,South,7991,"2,891,073.46","491,183.80","2,483,217.23","750,835.98",...,0% to 10%,2023-10-01,0.65,"1,580,096.88",0.64,0.09,0.01,(b) INR 15 to 25 lacs,(a) Var < 2%,Oct 2023
1,Oct-2023,Ahmedabad,Adams LLC,Active,South,14991,"7,275,275.56","2,071,406.09","5,328,267.81","2,517,247.77",...,0% to 10%,2023-10-01,0.52,"2,733,725.52",0.51,0.29,0.00,(e) Above INR 45 lacs,(a) Var < 2%,Oct 2023
2,Oct-2023,Ahmedabad,Adams LLC,Inactive,East,14545,"6,031,005.58","1,794,687.87","4,286,319.01","1,797,922.13",...,10% to 20%,2023-10-01,0.55,"2,331,269.06",0.54,0.25,0.00,(d) INR 35 to 45 lacs,(a) Var < 2%,Oct 2023
3,Oct-2023,Pune,"Aguilar, Becker and Hernandez",Active,East,11811,"3,561,403.95","868,767.88","2,765,074.26","1,211,271.41",...,More than 20%,2023-10-01,0.53,"1,428,381.52",0.52,-0.01,0.01,(c) INR 25 to 35 lacs,(a) Var < 2%,Oct 2023
4,Oct-2023,Ahmedabad,Aguirre-Goodman,Active,East,10219,"3,760,119.53","855,750.21","3,014,886.72","1,358,839.67",...,More than 20%,2023-10-01,0.50,"1,502,117.69",0.50,0.07,0.01,(c) INR 25 to 35 lacs,(a) Var < 2%,Oct 2023


Here, I check for missing values in the DataFrame to understand if there are any gaps in the data that might affect analysis or visualizations. This is a crucial step in data preparation to ensure the integrity of the insights derived from the dataset.

In [ ]:
df.isna().sum()

MONTH              0
CITY               0
STORE              0
STATUS             0
ZONE MAPPING       0
ORDER COUNT        0
CART SALES         0
DISCOUNT           0
NET REVENUE        0
IDEAL FOOD COST    0
GROSS MARGIN       0
KITCHEN EBITDA     0
VARIANCE           0
REVENUE COHORT     0
CM COHORT          0
EBITDA CATEGORY    0
EBITDA COHORT      0
MONTH_DT           0
GM %               0
CM                 0
CM %               0
EBITDA %           0
VARIANCE %         0
REVENUE BUCKET     0
VARIANCE BUCKET    0
MONTH LABEL        0
dtype: int64

## 2. Headline numbers

To get the value in small number, I divided the value by 1e7 (10000000).

In [5]:
headline = pd.Series({
    'Total net revenue (₹ Cr)' : df['NET REVENUE'].sum()    / 1e7,
    'Total GM (₹ Cr)'          : df['GROSS MARGIN'].sum()   / 1e7,
    'Total EBITDA (₹ Cr)'      : df['KITCHEN EBITDA'].sum() / 1e7,
    'Total variance (₹ Cr)'    : df['VARIANCE'].sum()       / 1e7,
    'Blended GM %'             : df['GROSS MARGIN'].sum()   / df['NET REVENUE'].sum() * 100,
    'Blended EBITDA %'         : df['KITCHEN EBITDA'].sum() / df['NET REVENUE'].sum() * 100,
    'Blended variance %'       : df['VARIANCE'].sum()       / df['NET REVENUE'].sum() * 100,
})
headline.round(2)

Total net revenue (₹ Cr)   736.12
Total GM (₹ Cr)            429.11
Total EBITDA (₹ Cr)        143.79
Total variance (₹ Cr)        4.24
Blended GM %                58.29
Blended EBITDA %            19.53
Blended variance %           0.58
dtype: float64

## 3. Dashboard 1 reproduction — Kitchen-level P&L (sample)

In [6]:
# Example: top 10 stores by Mar-2024 EBITDA
mar = df[df['MONTH'] == 'Mar-2024']
(mar.assign(**{'GM %': lambda x: x['GROSS MARGIN']/x['NET REVENUE'],
              'EBITDA %': lambda x: x['KITCHEN EBITDA']/x['NET REVENUE']})
    .nlargest(10, 'KITCHEN EBITDA')
    [['STORE','CITY','NET REVENUE','GM %','KITCHEN EBITDA','EBITDA %']])

,STORE,CITY,NET REVENUE,GM %,KITCHEN EBITDA,EBITDA %
1937,Long-Stokes,Hyderabad,"5,640,678.95",0.69,"2,591,394.39",0.46
1752,Adams LLC,Ahmedabad,"5,882,855.46",0.67,"2,544,886.12",0.43
2078,Walker-Mosley,Hyderabad,"5,529,660.35",0.69,"2,497,011.54",0.45
2083,"Washington, Wilson and Henry",Ahmedabad,"5,641,988.75",0.64,"2,281,691.57",0.40
1961,Miranda-Burch,Hyderabad,"5,336,444.32",0.68,"2,197,960.98",0.41
2040,"Smith, Mcclain and Edwards",Bangalore,"5,486,767.98",0.64,"2,179,528.24",0.40
1810,Carter-Meyers,Pune,"5,539,861.45",0.61,"2,110,193.58",0.38
1985,"Patterson, James and Allen",Bangalore,"5,216,458.57",0.66,"2,070,737.90",0.40
1776,Barnes PLC,Mumbai,"5,336,585.50",0.63,"2,011,642.37",0.38
1914,Jimenez LLC,Hyderabad,"4,718,069.31",0.65,"1,870,700.45",0.40


## 4. Dashboard 2 reproduction — Variance by revenue category

Average variance % per revenue bucket per month:

In [7]:
rev_order = [lab for lab, _, _ in REVENUE_BUCKETS]
month_order = df.sort_values('MONTH_DT')['MONTH LABEL'].drop_duplicates().tolist()

avg_var = (
    df.pivot_table(index='REVENUE BUCKET', columns='MONTH LABEL', values='VARIANCE %', aggfunc='mean')
      .reindex(index=rev_order, columns=month_order)
)
(avg_var * 100).round(2)

MONTH LABEL,Oct 2023,Nov 2023,Dec 2023,Jan 2024,Feb 2024,Mar 2024
REVENUE BUCKET,,,,,,
(a) Below INR 15 lacs,NaN,NaN,NaN,NaN,NaN,NaN
(b) INR 15 to 25 lacs,0.87,0.89,0.98,0.95,0.92,0.84
(c) INR 25 to 35 lacs,0.67,0.67,0.66,0.71,0.69,0.70
(d) INR 35 to 45 lacs,0.49,0.52,0.53,0.50,0.53,0.49
(e) Above INR 45 lacs,0.38,0.40,0.42,0.38,0.40,0.40


In [8]:
# Store count by revenue bucket × month
store_count = (
    df.pivot_table(index='REVENUE BUCKET', columns='MONTH LABEL', values='STORE',
                   aggfunc=pd.Series.nunique, fill_value=0)
      .reindex(index=rev_order, columns=month_order, fill_value=0)
)
store_count

MONTH LABEL,Oct 2023,Nov 2023,Dec 2023,Jan 2024,Feb 2024,Mar 2024
REVENUE BUCKET,,,,,,
(a) Below INR 15 lacs,0,0,0,0,0,0
(b) INR 15 to 25 lacs,43,56,41,45,45,48
(c) INR 25 to 35 lacs,126,141,147,138,139,134
(d) INR 35 to 45 lacs,132,97,117,107,120,112
(e) Above INR 45 lacs,48,53,43,58,44,52


## 5. Extra insights

### 5.1 Does variance correlate with EBITDA?

In [9]:
store_summary = (
    df.groupby('STORE')
      .apply(lambda g: pd.Series({
          'net_rev':    g['NET REVENUE'].sum(),
          'variance_%': g['VARIANCE'].sum()       / g['NET REVENUE'].sum(),
          'ebitda_%':   g['KITCHEN EBITDA'].sum() / g['NET REVENUE'].sum(),
      }), include_groups=False)
)
corr = store_summary['variance_%'].corr(store_summary['ebitda_%'])
print(f'Pearson correlation between Variance % and EBITDA %: {corr:+.3f}')
store_summary.describe()

Pearson correlation between Variance % and EBITDA %: -0.623


,net_rev,variance_%,ebitda_%
count,344.00,344.00,344.00
mean,"21,398,744.71",0.01,0.19
std,"3,699,633.63",0.00,0.05
min,"13,421,387.86",0.00,-0.06
25%,"19,416,189.42",0.01,0.16
50%,"21,079,795.70",0.01,0.19
75%,"22,508,874.75",0.01,0.23
max,"48,756,038.51",0.01,0.31


### 5.2 Zone-level performance

In [10]:
zone = (
    df.groupby('ZONE MAPPING')
      .apply(lambda g: pd.Series({
          'stores':     g['STORE'].nunique(),
          'net_rev_Cr': g['NET REVENUE'].sum() / 1e7,
          'gm_%':       g['GROSS MARGIN'].sum()   / g['NET REVENUE'].sum() * 100,
          'ebitda_%':   g['KITCHEN EBITDA'].sum() / g['NET REVENUE'].sum() * 100,
          'variance_%': g['VARIANCE'].sum()       / g['NET REVENUE'].sum() * 100,
      }), include_groups=False)
      .round(2)
      .sort_values('ebitda_%', ascending=False)
)
zone

,stores,net_rev_Cr,gm_%,ebitda_%,variance_%
ZONE MAPPING,,,,,
East,93.00,196.20,58.37,19.83,0.58
South,97.00,206.13,58.30,19.82,0.57
West,87.00,184.64,58.23,19.46,0.57
North,72.00,149.14,58.27,18.83,0.57


### 5.3 EBITDA negative stores — how many, and where?

In [11]:
neg = df[df['KITCHEN EBITDA'] < 0]
print(f'Loss-making rows: {len(neg):,} of {len(df):,} ({len(neg)/len(df):.1%})')
print(f'Unique loss-making stores: {neg["STORE"].nunique()}')
neg.groupby('CITY').agg(loss_rows=('STORE','count'),
                        stores=('STORE','nunique'),
                        total_loss_Cr=('KITCHEN EBITDA', lambda x: x.sum()/1e7)).round(2)

Loss-making rows: 241 of 2,100 (11.5%)
Unique loss-making stores: 184


,loss_rows,stores,total_loss_Cr
CITY,,,
Ahmedabad,59,46,-1.00
Bangalore,36,27,-0.56
Hyderabad,56,40,-0.91
Mumbai,45,36,-0.79
Pune,45,36,-0.82


### 5.4 Discount intensity vs profitability

In [12]:
df['DISCOUNT %'] = df['DISCOUNT'] / df['CART SALES']
df[['DISCOUNT %','EBITDA %','GM %']].corr().round(3)

,DISCOUNT %,EBITDA %,GM %
DISCOUNT %,1.00,-0.20,-0.01
EBITDA %,-0.20,1.00,0.49
GM %,-0.01,0.49,1.00


## 6. Findings summary

1. **Variance is tightly contained** — blended variance % sits well under 1% of net revenue across the six-month window, with most stores in the `(a) Var < 2%` bucket. Food wastage is **not** the primary EBITDA lever here.
2. **EBITDA % is driven mostly by gross margin and operating leverage**, not by variance.
3. **Loss-making stores** are concentrated in a small number of cities — useful for targeted intervention.
4. **Discount intensity has a measurable negative correlation** with EBITDA %, indicating discount-led top-line growth is margin-dilutive.
5. **Zone heat-map** shows clear winner / laggard zones — actionable for ops prioritisation.

All of these are interactively explorable in the Streamlit app (`Insights (bonus)` tab).